# 2.6 Juegos y adversarios: Minimax y poda alpha-beta

**Asignatura:** Introducción a la Inteligencia Artificial  
**Unidad 2:** Modelado y planteamiento de problemas

## Propósito

Implementar y comparar **minimax** y **poda alpha-beta** sobre el mismo árbol de juego.

Se analizarán:

a) Valor minimax

b) Mejor decisión para MAX

c) Número de nodos evaluados

d) Ramas podadas

e) Influencia del orden de exploración

> **Idea clave:** alpha-beta obtiene la misma decisión que minimax, pero puede evitar explorar ramas irrelevantes.

## 1. Árbol de juego

```text
                 MAX
              /       \
             A         B
             ↓         ↓
            MIN       MIN
           /   \     /   \
          4     7   2     9
```

Los valores de las hojas representan utilidades desde la perspectiva de MAX.

In [ ]:
arbol = {
    "raiz": ["A", "B"],
    "A": [4, 7],
    "B": [2, 9]
}

print(arbol)

## 2. Minimax

MAX selecciona el valor máximo y MIN selecciona el valor mínimo.

In [ ]:
contador_minimax = 0

def minimax(arbol_local, nodo, es_max):
    global contador_minimax
    contador_minimax += 1

    if isinstance(nodo, (int, float)):
        return nodo

    hijos = arbol_local[nodo]

    if es_max:
        return max(minimax(arbol_local, hijo, False) for hijo in hijos)
    else:
        return min(minimax(arbol_local, hijo, True) for hijo in hijos)

contador_minimax = 0
valor_minimax = minimax(arbol, "raiz", True)

valor_A = min(arbol["A"])
valor_B = min(arbol["B"])
mejor_accion = "A" if valor_A >= valor_B else "B"

print("Valor de A:", valor_A)
print("Valor de B:", valor_B)
print("Valor minimax:", valor_minimax)
print("Mejor acción para MAX:", mejor_accion)
print("Nodos evaluados:", contador_minimax)

### Resultado

```text
MIN(A) = min(4, 7) = 4
MIN(B) = min(2, 9) = 2
MAX(4, 2) = 4
```

Por tanto, MAX selecciona **A**.

## 3. Poda alpha-beta

Alpha-beta mantiene dos límites:

\[
\alpha = \text{mejor valor que MAX puede garantizar}
\]

\[
\beta = \text{mejor valor que MIN puede garantizar}
\]

Cuando:

\[
\alpha \geq \beta
\]

puede detenerse la exploración de la rama.

In [ ]:
contador_ab = 0
podas = []

def alpha_beta(arbol_local, nodo, es_max, alpha=float("-inf"), beta=float("inf")):
    global contador_ab, podas
    contador_ab += 1

    if isinstance(nodo, (int, float)):
        return nodo

    hijos = arbol_local[nodo]

    if es_max:
        valor = float("-inf")
        for i, hijo in enumerate(hijos):
            valor = max(valor, alpha_beta(arbol_local, hijo, False, alpha, beta))
            alpha = max(alpha, valor)

            if alpha >= beta:
                restantes = len(hijos) - (i + 1)
                if restantes > 0:
                    podas.append((nodo, restantes))
                break
        return valor

    else:
        valor = float("inf")
        for i, hijo in enumerate(hijos):
            valor = min(valor, alpha_beta(arbol_local, hijo, True, alpha, beta))
            beta = min(beta, valor)

            if alpha >= beta:
                restantes = len(hijos) - (i + 1)
                if restantes > 0:
                    podas.append((nodo, restantes))
                break
        return valor

contador_ab = 0
podas = []
valor_ab = alpha_beta(arbol, "raiz", True)

print("Valor alpha-beta:", valor_ab)
print("Nodos evaluados:", contador_ab)
print("Podas:", podas)

## 4. Interpretación de la poda

Después de evaluar A:

\[
\alpha = 4
\]

Al comenzar B, MIN encuentra primero el valor:

\[
2
\]

por lo que:

\[
\beta = 2
\]

Como:

\[
\beta \leq \alpha
\]

la hoja `9` ya no puede cambiar la decisión de MAX y puede podarse.

## 5. Comparación experimental

In [ ]:
import pandas as pd

comparacion = pd.DataFrame([
    {
        "Algoritmo": "Minimax",
        "Valor final": valor_minimax,
        "Mejor acción": mejor_accion,
        "Nodos evaluados": contador_minimax,
        "Podas": 0
    },
    {
        "Algoritmo": "Alpha-beta",
        "Valor final": valor_ab,
        "Mejor acción": mejor_accion,
        "Nodos evaluados": contador_ab,
        "Podas": len(podas)
    }
])

comparacion

## 6. Influencia del orden de exploración

La eficiencia de alpha-beta depende del orden de las ramas.

Compararemos:

```text
A → B
```

con:

```text
B → A
```

In [ ]:
def alpha_beta_stats(arbol_local, nodo, es_max, alpha=float("-inf"), beta=float("inf"), stats=None):
    if stats is None:
        stats = {"nodos": 0, "podas": 0}

    stats["nodos"] += 1

    if isinstance(nodo, (int, float)):
        return nodo, stats

    hijos = arbol_local[nodo]

    if es_max:
        valor = float("-inf")
        for i, hijo in enumerate(hijos):
            v, stats = alpha_beta_stats(arbol_local, hijo, False, alpha, beta, stats)
            valor = max(valor, v)
            alpha = max(alpha, valor)

            if alpha >= beta:
                stats["podas"] += len(hijos) - (i + 1)
                break
        return valor, stats

    else:
        valor = float("inf")
        for i, hijo in enumerate(hijos):
            v, stats = alpha_beta_stats(arbol_local, hijo, True, alpha, beta, stats)
            valor = min(valor, v)
            beta = min(beta, valor)

            if alpha >= beta:
                stats["podas"] += len(hijos) - (i + 1)
                break
        return valor, stats

arbol_AB = {
    "raiz": ["A", "B"],
    "A": [4, 7],
    "B": [2, 9]
}

arbol_BA = {
    "raiz": ["B", "A"],
    "A": [4, 7],
    "B": [2, 9]
}

valor_AB, stats_AB = alpha_beta_stats(arbol_AB, "raiz", True, stats={"nodos": 0, "podas": 0})
valor_BA, stats_BA = alpha_beta_stats(arbol_BA, "raiz", True, stats={"nodos": 0, "podas": 0})

pd.DataFrame([
    {
        "Orden": "A → B",
        "Valor": valor_AB,
        "Nodos evaluados": stats_AB["nodos"],
        "Podas": stats_AB["podas"]
    },
    {
        "Orden": "B → A",
        "Valor": valor_BA,
        "Nodos evaluados": stats_BA["nodos"],
        "Podas": stats_BA["podas"]
    }
])

## 7. Experimento

Modifica las hojas, por ejemplo:

```text
A = [3, 8]
B = [5, 6]
```

y vuelve a ejecutar alpha-beta.

In [ ]:
arbol_experimento = {
    "raiz": ["A", "B"],
    "A": [3, 8],
    "B": [5, 6]
}

valor_exp, stats_exp = alpha_beta_stats(
    arbol_experimento,
    "raiz",
    True,
    stats={"nodos": 0, "podas": 0}
)

print("Valor alpha-beta:", valor_exp)
print("Nodos evaluados:", stats_exp["nodos"])
print("Podas:", stats_exp["podas"])

## 8. Actividad de ampliación

Construye un árbol con al menos **tres niveles de decisión** y compara:

a) Valor minimax

b) Mejor acción

c) Nodos evaluados por minimax

d) Nodos evaluados por alpha-beta

e) Número de podas

f) Efecto del orden de exploración

---

## 9. Preguntas de análisis

a) ¿Por qué MAX elige la rama A en el ejemplo original?

b) ¿Por qué la hoja 9 puede podarse?

c) ¿Alpha-beta modifica el valor minimax?

d) ¿Qué representa \(lpha\)?

e) ¿Qué representa \(eta\)?

f) ¿Por qué el orden de exploración afecta la eficiencia?

g) ¿Qué ventaja práctica ofrece alpha-beta?

---

## Conclusión

> **Minimax determina la mejor decisión suponiendo que el adversario juega de forma óptima; alpha-beta conserva esa decisión reduciendo la exploración innecesaria.**

## Referencias

Russell, S. J., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. Capítulo 6: *Adversarial Search and Games*.

Knuth, D. E., & Moore, R. W. (1975). An analysis of alpha-beta pruning. *Artificial Intelligence, 6*(4), 293–326.